**Milestone 6 · chat app test.** This notebook ran on commits `a4cd567` / `c12faf5`, when the app lived in `app/` at the repo root. Since commit `e117c41` the app is the standalone [`deploy/`](https://github.com/jamalla/arabic-book-recommender-agent/tree/main/deploy) folder. To re-run it today: install `repo/deploy/requirements.txt` and use `sys.path.insert(0, "repo/deploy")`; the rest is unchanged. The live deployment is described in `deploy/README.md`.

In [1]:
# Milestone 6 · Step 1 — the app code (commit a4cd567) + keys from Kaggle Secrets (never printed)
!git clone -q https://github.com/jamalla/arabic-book-recommender-agent.git repo
!pip install -q -r repo/requirements.txt
import os, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = secrets.get_secret("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = secrets.get_secret("Public Key")  # the label used in Kaggle
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"
os.environ["LANGFUSE_TRACING_ENVIRONMENT"] = "development"  # test chats stay out of "production"
sys.path.insert(0, "repo")

import gradio as gr
from langfuse import get_client
print("gradio", gr.__version__, "| Langfuse connected:", get_client().auth_check())

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
gradio 6.26.0 | Langfuse connected: True


In [2]:
# Step 2 — test the app logic WITHOUT a browser: one chat with 2 turns, a 👍, the guards, and /health
import app.main as chat_app
from fastapi.testclient import TestClient

screen, messages, traces, session_id = chat_app.new_chat()
for question in ["أريد رواية تاريخية", "رشح لي واحدة أخرى من نفس النوع"]:  # turn 2 only makes sense with memory
    _, screen, messages, traces = chat_app.respond(question, messages, traces, session_id)
    print(f"👤 {question}\n🤖 {messages[-1]['content']}\n")

print("answer index -> trace id:", traces)
chat_app.feedback(traces, gr.LikeData(None, {"index": max(traces), "value": "", "liked": True}))  # 👍 on the last answer

print("too long ->", chat_app.respond("ك" * 600, messages, traces, session_id)[2][-1]["content"])
chat_app.MAX_QUESTIONS = 2  # pretend the limit is 2: a third question must be refused, without calling the model
print("limit    ->", chat_app.respond("سؤال ثالث", messages, traces, session_id)[2][-1]["content"])
chat_app.MAX_QUESTIONS = 15

print("/health  ->", TestClient(chat_app.create_server()).get("/health").json())
get_client().flush()
print("Langfuse session:", session_id)

2026-10-23 06:43:55,459 INFO langchain_openai.chat_models._client_utils: langchain-openai detected system proxy configuration and no explicit `http_socket_options` / `http_client` / `http_async_client` / `openai_proxy`; skipping the custom `httpx` transport so httpx's env-proxy auto-detection applies. Pass `http_socket_options=[...]` to opt back into kernel-level TCP keepalive tuning on top of the env proxy.


👤 أريد رواية تاريخية
🤖 لم أجد في الكتالوج رواية يوضح وصفها أنها تاريخية؛ هل تود أن أبحث لك عن رواية من التراث أو عن كتاب تاريخي؟

👤 رشح لي واحدة أخرى من نفس النوع
🤖 لم أجد رواية تاريخية واضحة في الكتالوج؛ يمكنني البحث لك عن رواية من التراث أو عن رواية تدور في حقبة قديمة.

answer index -> trace id: {1: 'da146724ad43fadf4e51b5892696d9d5', 3: 'df38fdda8ce55e5e1831c6a469676fa4'}
too long -> رسالتك طويلة جدًا. اكتب طلبك في أقل من 500 حرف من فضلك.
limit    -> وصلت إلى الحد الأقصى للأسئلة في هذه المحادثة. اضغط «محادثة جديدة» للبدء من جديد.


2026-10-23 06:44:22,762 INFO httpx2: HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


/health  -> {'status': 'ok', 'agent': 'langgraph', 'prompt': 'v2'}
Langfuse session: aaf8cd5d-f32a-470f-a6c6-6b7318619469


In [3]:
# Step 3 — was the refusal RIGHT? Check the evidence: what search_books saw, and the catalog itself
import pandas as pd
from bookrec.search import BookSearch

engine = BookSearch()
print("search_books('رواية تاريخية'):")
for b in engine.search("رواية تاريخية", k=5):
    print(f"   {b['score']:.2f}  {b['title']} — {b['category']} / {b.get('subcategory', '')}")

books = engine.books
fiction = books[books.category == "الأدب والخيال"]
historical = fiction[fiction.subcategory.str.contains("تاريخ", na=False) | fiction.description.str.contains("تاريخ", na=False)]
print(f"\nfiction books: {len(fiction)} | fiction books that mention تاريخ: {len(historical)}")
pd.set_option("display.max_colwidth", 120)
historical[["title", "author", "subcategory", "description"]]

search_books('رواية تاريخية'):
   0.18  تاريخ العرب القديم — التاريخ والجغرافيا / 
   0.16  تجليات جبل — الأدب والخيال / 
   0.15  المسرح في لبنان الشمالي منذ نشأته حتى نهاية القرن العشرين — الفنون / 
   0.13  التاريخ الخفي — التاريخ والجغرافيا / 
   0.13  عزلة الأحبة — الأدب والخيال / 

fiction books: 20 | fiction books that mention تاريخ: 2


,title,author,subcategory,description
10,تاريخ نظريات الاتصال,أرمان ماتلار ، ميشال ماتلار,تاريخ الأدب,يحيل مفهوم الاتصال على دلالات عديدة، وهو ليس بالأمر الجديد. إلا أن تكاثر التكنولوجيات الحديثة واتساع دائرة السارسات ...
12,ذكريات من القدس,سيرين الحسيني,السير والذكريات,إن كتاب سيرين الحسيني شهيد.. هو ذخيرة تاريخية وبشرية مؤلفة أساساً على شاكلة فسيفساء من شذرات ممتعة في معظمها، ومن مس...


In [4]:
# Step 4 — the real UI with the new examples (commit c12faf5), on a temporary public https link (*.gradio.live)
import importlib
!git -C repo pull -q
chat_app = importlib.reload(chat_app)
demo = chat_app.build_ui().queue(default_concurrency_limit=4)
demo.launch(share=True, inline=False, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://95ecdb40ca1307c3da.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# Step 5 — close the public link when the test is done: it runs on YOUR OpenAI key
demo.close()

Closing server running on port: 7860
